<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 03a — Custom Training & Batch Inference — ✅ Solution

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h00

</div>

## 📋 Lab Overview

In a production ML workflow, training typically doesn't happen in a notebook — it runs as a **managed job** on cloud infrastructure. Once a model is trained, it often needs to score large volumes of data offline rather than serving individual requests. Vertex AI provides **Custom Training Jobs** for scalable, reproducible training and **Batch Prediction** for high-throughput offline inference.

### Learning Objectives

1. Understand Vertex AI **pre-built containers** and how they simplify managed training.
2. Write a **training script** compatible with Vertex AI's conventions (`AIP_MODEL_DIR`).
3. Configure and submit a **`CustomTrainingJob`** to Vertex AI.
4. Prepare input data in **JSONL format** and upload it to Cloud Storage.
5. Launch a **`BatchPredictionJob`** and retrieve results from GCS.
6. Evaluate batch prediction results programmatically.

### Business Context

Imagine you are deploying an image classification model for an e-commerce platform. Product images arrive in bulk every night and need to be categorized before the catalog update. **Batch inference** is the right pattern here: high throughput, no latency constraint, and cost-efficient because resources are provisioned only for the duration of the job.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install dependencies, imports, GCP configuration |
| 1 | Custom Training Job | Pre-built containers, training script, submit job |
| 2 | Batch Prediction | Prepare JSONL input, launch job, retrieve results |
| 3 | Cleanup | Delete training job and batch prediction resources |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [ ]:
%pip install --upgrade --quiet google-cloud-aiplatform google-cloud-storage pillow numpy

### 0.2 Imports

In [ ]:
import os
import json
import warnings

import numpy as np
from PIL import Image

from google.cloud import aiplatform

warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Vertex AI SDK version: {aiplatform.__version__}")

### 0.3 Configuration

Replace the placeholders below with your own GCP project details. These constants will be reused in **Lab 03b** (online endpoints), so keep them consistent.

In [ ]:
##############################  TODO  ##############################
# ── Constants ──
PROJECT_ID = "your-project-id"          # @param {type:"string"}
LOCATION = "europe-west3"                # @param {type:"string"}
BUCKET_URI = "gs://your-bucket-name"    # @param {type:"string"}

# Set YOUR_NAME to a unique lowercase identifier (e.g. first letter of first name + last name).
YOUR_NAME = ...  # @param {type:"string"}
####################################################################

aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET_URI,
)
print(f"✅ Vertex AI initialized — project={PROJECT_ID}, location={LOCATION}")

> 💡 **Tip:** If you haven't created a Cloud Storage bucket yet, uncomment and run the cell below.

In [ ]:
# ! gsutil mb -l $LOCATION -p $PROJECT_ID $BUCKET_URI

---
## 1 · Custom Training Job

In Lab 03 you uploaded a pre-trained model to the Model Registry. Here, you will **train a model from scratch** using Vertex AI's managed training service. This means the training code runs on GCP infrastructure (not your local machine), and the resulting model artifacts are automatically saved to Cloud Storage.

### 1.1 Pre-built containers

Vertex AI offers **pre-built Docker containers** with popular ML frameworks (TensorFlow, PyTorch, scikit-learn, XGBoost). You don't need to write a Dockerfile — just pick a container that matches your framework and Python version, then provide your training script.

> 📖 **Docs:**
> - [Pre-built containers for training](https://cloud.google.com/vertex-ai/docs/training/pre-built-containers)
> - [Pre-built containers for prediction](https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers)

In [ ]:
# Training container: TensorFlow 2.17 CPU
TRAIN_IMAGE = "europe-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest"

# Prediction (serving) container: TensorFlow 2.15 CPU
DEPLOY_IMAGE = "europe-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest"

print(f"Training image:   {TRAIN_IMAGE}")
print(f"Prediction image: {DEPLOY_IMAGE}")

**✏️ Question 1 — Pre-built containers**

a) Why does Vertex AI use *separate* container images for training and prediction? What are the advantages of this separation?

b) When would you use a **custom container** instead of a pre-built one?

---
*✅ Solution:*

a) Training and serving have different requirements. Training containers include heavy dependencies for data loading, distributed training, and debugging (e.g., TensorBoard, `tensorflow_datasets`). Prediction containers are lightweight and optimized for low-latency inference — they only include the serving runtime and model loading code. Separating them reduces the serving container size, decreases cold-start time, and minimizes the attack surface in production.

b) You would use a custom container when: your framework isn't supported by pre-built containers (e.g., a custom C++ inference engine), you need specific library versions or OS-level dependencies, you want to bundle custom pre/post-processing logic inside the container, or you need to run non-Python code alongside the model.

---

### 1.2 The training script

The cell below writes a `task.py` file that defines and trains a CNN on the CIFAR-10 dataset. This script is designed to run **inside the pre-built container** on Vertex AI — not in your notebook.

Key conventions to notice:
- **`AIP_MODEL_DIR`**: an environment variable automatically set by Vertex AI. Your script must save the trained model to this path so Vertex AI can register it.
- **`argparse`**: training parameters (epochs, learning rate, etc.) are passed as command-line arguments. This makes the script configurable from the job submission.
- **Distribution strategies**: TensorFlow's `tf.distribute` API lets the same code run on 1 GPU, multiple GPUs, or multiple machines.

Read the script carefully — you will need to understand it for the questions that follow.

In [ ]:
%%writefile task.py
# Training script for CIFAR-10 image classification
# This script runs INSIDE a Vertex AI pre-built container.

import tensorflow_datasets as tfds
import tensorflow as tf
from tensorflow.python.client import device_lib
import argparse
import os
import sys

tfds.disable_progress_bar()

parser = argparse.ArgumentParser()
parser.add_argument('--lr', dest='lr', default=0.01, type=float,
                    help='Learning rate.')
parser.add_argument('--epochs', dest='epochs', default=10, type=int,
                    help='Number of epochs.')
parser.add_argument('--steps', dest='steps', default=200, type=int,
                    help='Number of steps per epoch.')
parser.add_argument('--distribute', dest='distribute', type=str,
                    default='single', help='Distribution strategy.')
args = parser.parse_args()

print(f"Python Version = {sys.version}")
print(f"TensorFlow Version = {tf.__version__}")
print(f"TF_CONFIG = {os.environ.get('TF_CONFIG', 'Not found')}")

# ── Distribution strategy ──
if args.distribute == 'single':
    if tf.test.is_gpu_available():
        strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    else:
        strategy = tf.distribute.OneDeviceStrategy(device="/cpu:0")
elif args.distribute == 'mirror':
    strategy = tf.distribute.MirroredStrategy()
elif args.distribute == 'multi':
    strategy = tf.distribute.experimental.MultiWorkerMirroredStrategy()

print(f"num_replicas_in_sync = {strategy.num_replicas_in_sync}")

# ── Dataset ──
BUFFER_SIZE = 10000
BATCH_SIZE = 64

def make_datasets_unbatched():
    def scale(image, label):
        image = tf.cast(image, tf.float32)
        image /= 255.0
        return image, label
    datasets, _ = tfds.load(name='cifar10', with_info=True, as_supervised=True)
    return datasets['train'].map(scale).cache().shuffle(BUFFER_SIZE).repeat()

# ── Model ──
def build_and_compile_cnn_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])
    model.compile(
        loss=tf.keras.losses.sparse_categorical_crossentropy,
        optimizer=tf.keras.optimizers.SGD(learning_rate=args.lr),
        metrics=['accuracy'],
    )
    return model

# ── Training ──
NUM_WORKERS = strategy.num_replicas_in_sync
GLOBAL_BATCH_SIZE = BATCH_SIZE * NUM_WORKERS
MODEL_DIR = os.getenv("AIP_MODEL_DIR")

train_dataset = make_datasets_unbatched().batch(GLOBAL_BATCH_SIZE)

with strategy.scope():
    model = build_and_compile_cnn_model()

model.fit(x=train_dataset, epochs=args.epochs, steps_per_epoch=args.steps)
model.export(MODEL_DIR)

**✏️ Question 2 — Training script conventions**

a) What is the role of the `AIP_MODEL_DIR` environment variable? What would happen if the script saved the model to a hardcoded local path instead?

b) The script supports three distribution strategies: `single`, `mirror`, and `multi`. In which scenario would you choose `mirror` over `single`?

---
*✅ Solution:*

a) `AIP_MODEL_DIR` is an environment variable set by Vertex AI at runtime. It points to a Cloud Storage path where the trained model artifacts should be saved. When the script calls `model.save(MODEL_DIR)`, Vertex AI automatically picks up the artifacts from this location and registers them as a `Model` resource. If you saved to a hardcoded local path (e.g., `./saved_model`), the artifacts would be lost when the training container shuts down, and Vertex AI would have no model to register.

b) `mirror` (MirroredStrategy) uses all GPUs on a single machine for data-parallel training. You would choose it over `single` when your training machine has multiple GPUs and you want to speed up training by distributing batches across them. `single` only uses one device. `multi` (MultiWorkerMirroredStrategy) goes further and distributes across multiple machines.

---

### 1.3 Configure training arguments

Before submitting the job, we define the hyperparameters and the distribution strategy to use. These values are passed to `task.py` as command-line arguments.

In [ ]:
JOB_NAME = "cifar10-custom-training"
MODEL_DIR = f"{BUCKET_URI}/{YOUR_NAME}/{JOB_NAME}"

EPOCHS = 20
STEPS = 100
TRAIN_STRATEGY = "single"

CMDARGS = [
    f"--epochs={EPOCHS}",
    f"--steps={STEPS}",
    f"--distribute={TRAIN_STRATEGY}",
]

print(f"Job name:  {JOB_NAME}")
print(f"Model dir: {MODEL_DIR}")
print(f"Args:      {CMDARGS}")

### 1.4 Submit the Custom Training Job

Use `CustomTrainingJob` to package and submit the training script. The `run()` method starts the job and blocks until it completes.

> 📖 **Docs:**
> - [`CustomTrainingJob`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.CustomTrainingJob)
> - [`CustomTrainingJob.run()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.CustomTrainingJob#google_cloud_aiplatform_CustomTrainingJob_run)

In [ ]:
# ✅ SOLUTION
job = aiplatform.CustomTrainingJob(
    display_name=JOB_NAME,
    script_path="task.py",
    container_uri=TRAIN_IMAGE,
    requirements=["tensorflow_datasets==4.9.2"],
    model_serving_container_image_uri=DEPLOY_IMAGE,
)

MODEL_DISPLAY_NAME = "cifar10-model"

model = job.run(
    model_display_name=MODEL_DISPLAY_NAME,
    args=CMDARGS,
    replica_count=1,
    machine_type="n1-standard-4",
    base_output_dir=MODEL_DIR,
)

> ⏳ **Training takes ~10–15 minutes.** While you wait, open the [Vertex AI Training console](https://console.cloud.google.com/vertex-ai/training/training-pipelines) in your browser and monitor the job status.

> 💡 **Tip:** Notice how the console shows the training logs streamed from the container. This is the same output you would see if running `task.py` locally.

In [ ]:
# Verify the model was created
print(f"✅ Model resource name: {model.resource_name}")
print(f"   Model display name:  {model.display_name}")
print(f"   Model URI:           {model.uri}")

**✏️ Question 3 — Custom Training Jobs**

a) In the GCP console, find the training job you just submitted. What **machine type** was used? How much did this job cost approximately? (Check the [pricing page](https://cloud.google.com/vertex-ai/pricing#custom-trained_models).)

b) If you needed to train a much larger model (e.g., a ResNet-50) on the full CIFAR-10 dataset, what changes would you make to the job configuration? Think about `machine_type`, `accelerator_type`, and `distribute`.

---
*✅ Solution:*

a) The machine type is `n1-standard-4` (4 vCPUs, 15 GB RAM). At current pricing (~$0.19/hour for n1-standard-4 in us-central1), a 15-minute training job costs approximately $0.05. This is very inexpensive because we used CPU-only and a small model. GPU training (e.g., NVIDIA_TESLA_T4) would cost ~$0.35/hour per GPU.

b) For a larger model, you would: use `machine_type="n1-standard-8"` or `n1-highmem-8"` for more CPU/RAM; add `accelerator_type="NVIDIA_TESLA_T4"` and `accelerator_count=1` (or more) for GPU training; change `distribute="mirror"` to use all GPUs on the machine; increase `EPOCHS` and `STEPS`; potentially use `replica_count > 1` with `distribute="multi"` for multi-machine training.

---

---
## 2 · Batch Prediction

Now that the model is trained and registered, we can use it for **batch inference**. Batch prediction is ideal when you need to score a large dataset asynchronously — for instance, classifying thousands of images overnight.

Vertex AI manages the compute infrastructure: it spins up prediction nodes, distributes the input data, collects results, and writes them to Cloud Storage.

### 2.1 Download test data

We use a small set of CIFAR-10 test images hosted in a public GCS bucket. The file naming convention encodes the true label: `image_{label}_{index}.jpg`.

In [ ]:
# Download test images from public GCS bucket
! gsutil -m cp -r gs://cloud-samples-data/ai-platform-unified/cifar_test_images .

print("✅ Test images downloaded.")
! ls cifar_test_images/

### 2.2 Load and preprocess test images

The images need to match the format expected by the model: normalized float32 values in [0, 1].

> 📖 **Docs:** [Batch prediction input formats](https://cloud.google.com/vertex-ai/docs/predictions/batch-predictions#batch_request_input)

In [ ]:
# ✅ SOLUTION
IMAGE_DIRECTORY = "cifar_test_images"

image_files = [f for f in os.listdir(IMAGE_DIRECTORY) if f.endswith(".jpg")]
image_data = [np.asarray(Image.open(os.path.join(IMAGE_DIRECTORY, f))) for f in image_files]
x_test = [(img / 255.0).astype(np.float32).tolist() for img in image_data]
y_test = [int(f.split("_")[1]) for f in image_files]

print(f"✅ Loaded {len(x_test)} images")
print(f"   Image shape: {np.array(image_data[0]).shape}")
print(f"   Labels: {y_test}")

### 2.3 Prepare JSONL input

Vertex AI Batch Prediction accepts several input formats. We use **JSONL** (JSON Lines): one JSON object per line, where each object is a single input instance. The file is uploaded to Cloud Storage so the batch job can read it.

In [ ]:
# ✅ SOLUTION
BATCH_PREDICTION_INSTANCES_FILE = "batch_prediction_instances.jsonl"
BATCH_PREDICTION_GCS_SOURCE = (
    f"{BUCKET_URI}/{YOUR_NAME}/batch_prediction_instances/{BATCH_PREDICTION_INSTANCES_FILE}"
)

# Write JSONL locally
with open(BATCH_PREDICTION_INSTANCES_FILE, "w") as f:
    for x in x_test:
        f.write(json.dumps(x) + "\n")

# Upload to GCS
! gsutil cp $BATCH_PREDICTION_INSTANCES_FILE $BATCH_PREDICTION_GCS_SOURCE

print(f"✅ Uploaded instances to: {BATCH_PREDICTION_GCS_SOURCE}")

### 2.4 Launch batch prediction

Now we call `model.batch_predict()` to start the batch prediction job. Vertex AI provisions compute nodes, loads the model, scores all instances, and writes results to GCS.

> 📖 **Docs:** [`Model.batch_predict()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.Model#google_cloud_aiplatform_Model_batch_predict)

In [ ]:
# ✅ SOLUTION
DESTINATION_FOLDER = "batch_prediction_results"
BATCH_PREDICTION_GCS_DEST_PREFIX = f"{BUCKET_URI}/{YOUR_NAME}/{DESTINATION_FOLDER}"

batch_prediction_job = model.batch_predict(
    instances_format="jsonl",
    predictions_format="jsonl",
    job_display_name="cifar10-batch-prediction",
    gcs_source=BATCH_PREDICTION_GCS_SOURCE,
    gcs_destination_prefix=BATCH_PREDICTION_GCS_DEST_PREFIX,
    machine_type="n1-standard-4",
    starting_replica_count=1,
    max_replica_count=1,
    sync=True,
)

> ⏳ **Batch prediction takes ~10–15 minutes.** You can monitor it in the [Batch Predictions console](https://console.cloud.google.com/vertex-ai/batch-predictions).

### 2.5 Retrieve and evaluate results

Once the job completes, results are written to GCS as JSONL files. Each line contains the model's prediction for one input instance.

In [ ]:
# Download results from GCS
RESULTS_DIRECTORY = "prediction_results"
RESULTS_DIRECTORY_FULL = RESULTS_DIRECTORY + "/" + DESTINATION_FOLDER
os.makedirs(RESULTS_DIRECTORY, exist_ok=True)

! gsutil -m cp -r $BATCH_PREDICTION_GCS_DEST_PREFIX $RESULTS_DIRECTORY

print("✅ Results downloaded.")

In [ ]:
# Parse the prediction results
latest_directory = max(
    (os.path.join(RESULTS_DIRECTORY_FULL, d)
     for d in os.listdir(RESULTS_DIRECTORY_FULL)),
    key=os.path.getmtime,
)

results_files = []
for dirpath, subdirs, files in os.walk(latest_directory):
    for file in files:
        if file.startswith("prediction.results"):
            results_files.append(os.path.join(dirpath, file))

results = []
for results_file in results_files:
    with open(results_file, "r") as file:
        results.extend([json.loads(line) for line in file.readlines()])

print(f"✅ Loaded {len(results)} prediction results")

In [ ]:
# ✅ SOLUTION
y_predicted = [np.argmax(result["prediction"]) for result in results]

correct = sum(np.array(y_predicted) == np.array(y_test))
total = len(y_predicted)
accuracy = correct / total

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {accuracy:.2%}")

**✏️ Question 4 — Batch prediction**

a) What input formats does Vertex AI Batch Prediction support besides JSONL? In which situation would you use BigQuery as input/output instead of GCS?

b) The batch job ran with `max_replica_count=1`. If you had 100,000 images to score, what parameter would you change, and how does Vertex AI distribute the workload across replicas?

---
*✅ Solution:*

a) Vertex AI Batch Prediction supports: JSONL, CSV, BigQuery, TF Record, TF Record GZIP, and file-list. BigQuery is ideal when your data is already in a data warehouse (e.g., structured features for tabular models), when you want results written directly to tables for downstream SQL analysis, or when you need to join predictions with other business data without an ETL step.

b) You would increase `max_replica_count` (e.g., to 5 or 10). Vertex AI automatically shards the input data across replicas — each replica receives a subset of instances to score. The results are merged into the output location. This provides near-linear scaling: 10 replicas can process ~10x the data in the same time. You should also consider increasing `starting_replica_count` to avoid the autoscaling ramp-up delay.

---

---
## 3 · Cleanup

Always clean up cloud resources after a lab to avoid unnecessary charges.

> ⚠️ **Important:** Do **not** delete the model if you plan to continue with **Lab 03b** (online endpoints). Only delete the training job and batch prediction job.

In [ ]:
# Delete the training job (does not delete the model)
try:
    job.delete()
    print("✅ Training job deleted.")
except Exception as e:
    print(f"Training job cleanup: {e}")

In [ ]:
# Delete the batch prediction job
try:
    batch_prediction_job.delete()
    print("✅ Batch prediction job deleted.")
except Exception as e:
    print(f"Batch prediction cleanup: {e}")

In [ ]:
# Print model info for Lab 06
print("═" * 60)
print("Keep these values for Lab 06:")
print(f"  MODEL_DISPLAY_NAME = \"{MODEL_DISPLAY_NAME}\"")
print(f"  model.resource_name = \"{model.resource_name}\"")
print("═" * 60)

---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| Setup | Configured GCP project and pre-built containers | `aiplatform.init()`, container URIs |
| Custom Training | Submitted a training script as a managed job | `CustomTrainingJob`, `job.run()` |
| Batch Prediction | Scored test data offline via a batch job | `model.batch_predict()`, JSONL, GCS |
| Cleanup | Deleted training and batch prediction resources | `job.delete()` |

**Next lab:** In **Lab 06**, you will deploy the same model behind an **online endpoint** for real-time predictions, configure **traffic splitting** for A/B testing, wrap the endpoint with **FastAPI**, and run **load tests** with Locust.